In [ ]:
# ## Import libraries
import glob
import joblib
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# ## Set up variables
DIR_DATA = Path("/content/data")
DIR_TRAIN = DIR_DATA / "train"
DIR_TEST = DIR_DATA / "test"

DIR_OUT_MODEL = Path("6_rf_models")
SEED = 42
WETLAND_CMAP = ListedColormap(["white", "tab:blue"])

In [ ]:
# ## Set up functions
def load_npy_pairs(img_dir, mask_dir):
    img_files = sorted(glob.glob(str(img_dir / "*.npy")))
    mask_files = sorted(glob.glob(str(mask_dir / "*.npy")))

    img_dict = {}
    mask_dict = {}

    # Build dictionaries
    for img in img_files:
        name = Path(img).stem  # train_001_003_p0
        img_dict[name] = img

    for msk in mask_files:
        name = Path(msk).stem.replace("_gt", "")  # remove _gt
        mask_dict[name] = msk

    # Match pairs
    pairs = []
    for name in img_dict:
        if name in mask_dict:
            pairs.append((img_dict[name], mask_dict[name]))
        else:
            print(f"Mask missing for {name}")

    return pairs

def npy_pairs_to_pixels(pairs):
    X_list = []
    y_list = []

    for img_path, mask_path in pairs:
        img = np.load(img_path)      # (6, H, W)
        mask = np.load(mask_path)[0] # (H, W)

        img = img.astype(np.float32) / 10000.0

        H, W = mask.shape
        X = img.reshape(6, -1).T
        y = mask.reshape(-1)

        X_list.append(X)
        y_list.append(y)

    X_all = np.vstack(X_list)
    y_all = np.hstack(y_list)

    return X_all, y_all

def balanced_sample(X, y, n_samples=400000):
    wetland_idx = np.where(y == 1)[0]
    nonwetland_idx = np.where(y == 0)[0]

    n_each = n_samples // 2

    wetland_sample = np.random.choice(wetland_idx, n_each, replace=False)
    nonwetland_sample = np.random.choice(nonwetland_idx, n_each, replace=False)

    idx = np.concatenate([wetland_sample, nonwetland_sample])

    return X[idx], y[idx]

def iou_score(y_true, y_pred):
    intersection = np.sum((y_true == 1) & (y_pred == 1))
    union = np.sum((y_true == 1) | (y_pred == 1))
    return intersection / (union + 1e-6)

def find_best_threshold(probs, labels):
    thresholds = np.linspace(0.1, 0.9, 17)
    best_thr = 0.5
    best_iou = 0

    for t in thresholds:
        preds = (probs > t).astype(np.uint8)
        iou = iou_score(labels, preds)

        if iou > best_iou:
            best_iou = iou
            best_thr = t

    return best_thr, best_iou

def show_prediction_rf(rf, img_path, mask_path):
    img = np.load(img_path)      # (6, H, W)
    mask = np.load(mask_path)[0] # (H, W)

    # Normalize like training
    img_norm = img.astype(np.float32) / 10000.0

    H, W = mask.shape

    # Reshape to pixel table
    X = img_norm.reshape(6, -1).T

    # Predict
    pred = rf.predict(X)
    pred = pred.reshape(H, W)

    # Make RGB from bands (B4, B3, B2)
    rgb = img[[2,1,0]]
    rgb = np.transpose(rgb, (1,2,0))
    rgb = rgb / rgb.max()

    # Custom colormap: 0 = white, 1 = blue
    wetland_cmap = ListedColormap(["white", "tab:blue"])


    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(rgb)
    plt.title("Image")

    plt.subplot(1,3,2)
    plt.imshow(mask, cmap=WETLAND_CMAP)
    plt.title("Ground Truth")

    plt.subplot(1,3,3)
    plt.imshow(pred, cmap=WETLAND_CMAP)
    plt.title("RF Prediction")

    plt.show()


In [ ]:
# ## Set up datasets
train_pairs_full = load_npy_pairs(
    DIR_TRAIN / "images",
    DIR_TRAIN / "masks_binary"
)

test_pairs = load_npy_pairs(
    DIR_TEST / "images",
    DIR_TEST / "masks_binary"
)

# Split 80:20
np.random.seed(SEED)
indices = np.random.permutation(len(train_pairs_full))

train_size = int(0.8 * len(train_pairs_full))

train_pairs = [train_pairs_full[i] for i in indices[:train_size]]
val_pairs   = [train_pairs_full[i] for i in indices[train_size:]]

print("Train pairs:", len(train_pairs))
print("Val pairs:", len(val_pairs))
print("Test pairs:", len(test_pairs))

X_train, y_train = npy_pairs_to_pixels(train_pairs)
X_val, y_val = npy_pairs_to_pixels(val_pairs)
X_test, y_test = npy_pairs_to_pixels(test_pairs)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)
print("Test shape:", X_test.shape)

X_train_s, y_train_s = balanced_sample(X_train, y_train, 400000)
X_val_s, y_val_s = balanced_sample(X_val, y_val, 100000)

In [ ]:
## Train Random Forest and tune the hyper parameters
param_grid = {
    "n_estimators": [150, 200, 250],
    "max_depth": [15, 20, 25],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt"]
}

best_iou = 0
best_params = None
best_model = None

for n in param_grid["n_estimators"]:
    for d in param_grid["max_depth"]:
        for leaf in param_grid["min_samples_leaf"]:
            for mf in param_grid["max_features"]:

                rf = RandomForestClassifier(
                    n_estimators=n,
                    max_depth=d,
                    min_samples_leaf=leaf,
                    max_features=mf,
                    criterion="gini",
                    n_jobs=-1,
                    random_state=SEED,
                    verbose=1
                )

                rf.fit(X_train_s, y_train_s)

                y_val_pred = rf.predict(X_val_s)
                val_iou = iou_score(y_val_s, y_val_pred)

                print(f"Trees={n}, Depth={d}, Leaf={leaf}, IoU={val_iou:.4f}")

                if val_iou > best_iou:
                    best_iou = val_iou
                    best_params = (n, d, leaf, mf)
                    best_model = rf


print("Best Params:", best_params)
print("Best Validation IoU:", best_iou)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename_out = f"rf_{timestamp}" + \
    f"_n{best_params[0]}" + \
    f"_d{best_params[1]}" + \
    f"_l{best_params[2]}" + \
    f"_mf{best_params[3]}" + \
    ".pkl"
joblib.dump(best_model, DIR_OUT_MODEL / filename_out)
print("RF model saved")

In [ ]:
# ### 2021 Summer ### #
# Trees=50, Depth=10, Leaf=1, IoU=0.7185
# Trees=50, Depth=10, Leaf=2, IoU=0.7194
# Trees=50, Depth=10, Leaf=5, IoU=0.7194

# Trees=50, Depth=15, Leaf=1, IoU=0.7387
# Trees=50, Depth=15, Leaf=2, IoU=0.7383
# Trees=50, Depth=15, Leaf=5, IoU=0.7397

# Trees=50, Depth=20, Leaf=1, IoU=0.7427
# Trees=50, Depth=20, Leaf=2, IoU=0.7440
# Trees=50, Depth=20, Leaf=5, IoU=0.7424

# Trees=100, Depth=10, Leaf=1, IoU=0.7190
# Trees=100, Depth=10, Leaf=2, IoU=0.7199
# Trees=100, Depth=10, Leaf=5, IoU=0.7195

# Trees=100, Depth=15, Leaf=1, IoU=0.7393
# Trees=100, Depth=15, Leaf=2, IoU=0.7400
# Trees=100, Depth=20, Leaf=1, IoU=0.7434

# Trees=100, Depth=20, Leaf=1, IoU=0.7434
# Trees=100, Depth=20, Leaf=2, IoU=0.7446
# Trees=100, Depth=20, Leaf=5, IoU=0.7429

# Trees=150, Depth=20, Leaf=2, IoU=0.7448
# Trees=150, Depth=25, Leaf=2, IoU=0.7437

# Trees=200, Depth=20, Leaf=2, IoU=0.7456
# Trees=200, Depth=25, Leaf=2, IoU=0.7443

# Trees=250, Depth=20, Leaf=2, IoU=0.7452

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename_out = f"rf_{timestamp}" + \
    f"_n{best_params[0]}" + \
    f"_d{best_params[1]}" + \
    f"_l{best_params[2]}" + \
    f"_mf{best_params[3]}" + \
    ".pkl"
joblib.dump(best_model, DIR_OUT_MODEL / filename_out)
print("RF model saved")

In [ ]:
# ### 2025 ### #
# Trees=50, Depth=10, Leaf=1, IoU=0.7844
# Trees=50, Depth=10, Leaf=2, IoU=0.7850
# Trees=50, Depth=10, Leaf=5, IoU=0.7848

# Trees=50, Depth=15, Leaf=1, IoU=0.7926
# Trees=50, Depth=15, Leaf=2, IoU=0.7929
# Trees=50, Depth=15, Leaf=5, IoU=0.7929

# Trees=50, Depth=20, Leaf=1, IoU=0.7935
# Trees=50, Depth=20, Leaf=2, IoU=0.7937
# Trees=50, Depth=20, Leaf=5, IoU=0.7928

# Trees=100, Depth=10, Leaf=1, IoU=0.7842
# Trees=100, Depth=10, Leaf=2, IoU=0.7848
# Trees=100, Depth=10, Leaf=5, IoU=0.7852

# Trees=100, Depth=15, Leaf=1, IoU=0.7936
# Trees=100, Depth=15, Leaf=2, IoU=0.7936
# Trees=100, Depth=15, Leaf=5, IoU=0.7931

# Trees=100, Depth=20, Leaf=1, IoU=0.7942
# Trees=100, Depth=20, Leaf=2, IoU=0.7946
# Trees=100, Depth=20, Leaf=5, IoU=0.7936

In [ ]:
param_grid = {
    "n_estimators": [200],
    "max_depth": [20],
    "min_samples_leaf": [2],
    "max_features": ["sqrt"]
}

best_iou = 0
best_params = None
best_model = None
best_threshold = 0.5

start_time = time.time()
for n in param_grid["n_estimators"]:
    for d in param_grid["max_depth"]:
        for leaf in param_grid["min_samples_leaf"]:
            for mf in param_grid["max_features"]:

                rf = RandomForestClassifier(
                    n_estimators=n,
                    max_depth=d,
                    min_samples_leaf=leaf,
                    max_features=mf,
                    criterion="gini",
                    n_jobs=-1,
                    random_state=SEED,
                    verbose=0
                )

                rf.fit(X_train_s, y_train_s)

                # Get probabilities instead of predictions
                val_probs = rf.predict_proba(X_val_s)[:, 1]

                # Find best threshold for this model
                thr, val_iou = find_best_threshold(val_probs, y_val_s)

                print(f"Trees={n}, Depth={d}, Leaf={leaf}, Thr={thr:.2f}, IoU={val_iou:.4f}")

                if val_iou > best_iou:
                    best_iou = val_iou
                    best_params = (n, d, leaf, mf)
                    best_model = rf
                    best_threshold = thr

end_time = time.time()
print(f"Elapsed time: {end_time - start_time} seconds")

print("Best Params:", best_params)
print("Best Threshold:", best_threshold)
print("Best Validation IoU:", best_iou)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename_out = f"rf_{timestamp}" + \
    f"_n{best_params[0]}" + \
    f"_d{best_params[1]}" + \
    f"_l{best_params[2]}" + \
    f"_mf{best_params[3]}" + \
    ".pkl"
joblib.dump(best_model, DIR_OUT_MODEL / filename_out)
print("RF model saved")

In [ ]:
# Trees=200, Depth=20, Leaf=2, Thr=0.35, IoU=0.7591
# Best Params: (200, 20, 2, 'sqrt')
# Best Threshold: 0.35
# Best Validation IoU: 0.7590981012527673
# RF model saved

In [ ]:
best_model = joblib.load(
    DIR_OUT_MODEL / "rf_20260328_224937_n200_d20_l2_mfsqrt.pkl"
)
print("RF model loaded")

In [ ]:
# Use probability + threshold
probs_test = best_model.predict_proba(X_test)[:, 1]
y_pred_test = (probs_test > best_threshold).astype(np.uint8)

test_iou_rf = iou_score(y_test, y_pred_test)

print("Random Forest Test IoU:", test_iou_rf)

In [ ]:
y_pred_test = best_model.predict(X_test)
test_iou_rf = iou_score(y_test, y_pred_test)

print("Random Forest Test IoU:", test_iou_rf)

In [ ]:
# ### 2021 Summer ### #
# rf_20260328_224937_n200_d20_l2_mfsqrt.pkl
# Random Forest Test IoU: 0.6399948940875977
# Threshold Alignement: 0.5832246842777599

# ### 2025 ### #
# rf_20260322_171721_n100_d20_l2_mfsqrt.pkl
# Random Forest Test IoU: 0.6199460474246523

In [ ]:
img_path, mask_path = test_pairs[0]
show_prediction_rf(best_model, img_path, mask_path)